# Observing System Experiment (OSE)

In auroral imager context, an OSE simulates what a space-based imager would see from a low Earth orbit (LEO) platform. In short, it allows us to see the AIC mosaic "scene" that the imager will see given a time cadence, pixel resolution, and field of view.

<div class="alert alert-block alert-info">
<b>Note:</b>
OSEs are computationally expensive and each one will take about an hour to run. Plenty of time for a coffee break!
</div>

In [ ]:
import pathlib
import warnings
import itertools
from collections import namedtuple
from datetime import datetime

import cartopy.crs
import cartopy.feature as cfeature
import matplotlib.font_manager
import matplotlib.textpath
import matplotlib.path
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.gridspec as gridspec
try:
    import fontawesome
    fontawesome_imported = True
except ImportError:
    fontawesome_imported = False

from asilib.mission import Example_Satellite
import asilib


If you need a nicer marker for the satellite you'll need to install fontawesome and donwload the Font Awesome `.otf` file (free tier) from their [website](https://fontawesome.com/download). Unzip it and set the fontawesome_path to that directory.

In [ ]:
fontawesome_path = pathlib.Path.cwd() / "Font Awesome 7 Free-Solid-900.otf"
warned = False

def getmarker(mID):
    global warned
    if not fontawesome_imported:
        return "s"
    elif not fontawesome_path.exists():
        if not warned:
            warnings.warn(
                f"A fontawesome font file not found at {fontawesome_path}. Using default marker. "
                f"Download them from https://fontawesome.com/download, unzip the archive, and set "
                f"the fontawesome_path to the font file.")
            warned = True
        return "s"
        
    symbol = fontawesome.icons[mID]
    fp = matplotlib.font_manager.FontProperties(fname=fontawesome_path)

    v, codes = matplotlib.textpath.TextToPath().get_text_path(fp, symbol)
    v = np.array(v)
    mean = np.mean([np.max(v,axis=0), np.min(v, axis=0)], axis=0)
    return matplotlib.path.Path(v-mean, codes, closed=False)

## CINEMA constellation OSE of a turning streamer

### CINEMA constellation parameters and ephemeris
Which is Event #2 in the [Ohtani+2022](https://doi.org/10.1029/2021JA030114) paper. We begin by specifying the CINEMA constallation with three rows of three satelites, separated by 1 hour in MLT. We will call  the ephemeris in latitude, longitude, altitude (LLA) ephemeris coordinates. 

In [ ]:
in_track_separation_minutes = 5
orbit_period_minutes = 95  # a good guess, but this can be more accurately calculated.
mean_anomaly_deg = 35  # Sets the orbit phase to begin above the high latitude sector.
sat_alt = 600  # in kilometers
center_ltan = 2.25  # Similar to MLT.
delta_mean_anomaly_deg = 360*in_track_separation_minutes/orbit_period_minutes

Now let's set the CINEMA-AIM orientation parameters

In [ ]:
fov = (55, 65) # (in-track, cross-track) field of view in degrees
pixel_resolution = (124, 124)
roll = 7  # A rotation of the imager boresight in degrees, positive is clockwise. If it is zero, the imager boresight is aligned with north (not the in-track direction).
ona = 0  # We can also tilt the FOV away from zenith by the off-nadir angle, but this is not necessary for this example.
azimuth = 0  # tilt the FOV by ona in the azimuthal direction (defined as 0 is north)

Let's put it all together to generate the CINEMA constellation ephemeris using skyfield and the sgp4 orbit propagator (called by Example_Satellite)

In [ ]:
time_range = (datetime(2008, 2, 4, 10, 35), datetime(2008, 2, 4, 10, 55))
aurora_alt = 110

orbit_parameter_tuple_type = namedtuple(
    'orbit_parameter_tuple_type', 
    ['mean_anomaly_deg', 'ltan_hours', 'alt_km']
    )

ltan_hours = [center_ltan-1, center_ltan, center_ltan+1]
mean_anomalies = [
    mean_anomaly_deg+delta_mean_anomaly_deg, 
    mean_anomaly_deg, 
    mean_anomaly_deg-delta_mean_anomaly_deg
    ]
constellation = {
    i:orbit_parameter_tuple_type(
        mean_anomaly_deg=mean_anomaly, 
        ltan_hours=ltan,
        alt_km=sat_alt,
        ) for i, (mean_anomaly, ltan) in enumerate(itertools.product(mean_anomalies, ltan_hours))
    }

ephemeris = [None, None]
for key, value in constellation.items():
    ephemeris_obj = Example_Satellite(
        cadence_s=0.5,
        time_range=time_range,
        mean_anomaly_deg=value.mean_anomaly_deg,
        ltan_hours=value.ltan_hours,
        altitude_km=value.alt_km,
    )
    sat_ephemeris = ephemeris_obj.ephemeris()
    if ephemeris[0] is None:
        ephemeris[0] = sat_ephemeris[0]
        ephemeris[1] = sat_ephemeris[1].reshape(*sat_ephemeris[1].shape, 1)
    else:
        ephemeris[1] = np.concatenate(
            (ephemeris[1], sat_ephemeris[1].reshape(*sat_ephemeris[1].shape, 1)), axis=2
            )

### Initialize the ASI mosaic

In [ ]:
location_codes = [
    'FYKN',
    'INUV',
    'FSIM',
    'WHIT',
    'KIAN',
    ]

asis = asilib.Imagers(
    [asilib.asi.themis(code, time_range=time_range, alt=aurora_alt) for code in location_codes]
    )

### Create figure panels and run OSE

We first need to create the main mosaic plot and the nine CINEMA-AIM subplots, which are passed into `OSE()` via the `ax` and `bx` kwargs, respetively.

In [ ]:
fig = plt.figure(figsize=(4, 7.5))
gs = gridspec.GridSpec(nrows=4, ncols=3, figure=fig, height_ratios=(3, 1, 1, 1))

center = (
    np.mean(asis.lon_bounds), np.mean(asis.lat_bounds)
)
projection = cartopy.crs.Orthographic(
    central_longitude=center[0], 
    central_latitude=center[1]
)

ax = fig.add_subplot(gs[0, :], projection=projection)
ax.add_feature(cfeature.LAND, color='grey')
ax.add_feature(cfeature.OCEAN, color='cyan')
ax.add_feature(cfeature.COASTLINE, edgecolor='k')
ax.gridlines(linestyle=':')
ax.set_global()
ax.set_extent(
    (center[0]-20, center[0]+20, center[1]-10, center[1]+8), 
    crs=cartopy.crs.PlateCarree()
    )

bx = np.nan*np.zeros((3, 3), dtype=object)
for i in range(3):
    for j in range(3):
        bx[i, j] = fig.add_subplot(gs[i+1, j])
        bx[i, j].set_aspect('equal')
        bx[i, j].xaxis.set_visible(False)
        bx[i, j].yaxis.set_visible(False)

ose = asilib.OSE(
    asis, 
    ephemeris, 
    fov=fov, 
    pixel_resolution=pixel_resolution, 
    roll=roll, 
    ona=ona, 
    azimuth=azimuth
    )
plt.suptitle(
    f'CINEMA OSE | fov={fov} [deg]\n'
    f'alt={sat_alt} [km] | resolution={pixel_resolution} [px]', 
    fontsize=12
    )
plt.subplots_adjust(
    bottom=0.01, top=0.95, left=0.01, right=0.99, wspace=0.03, hspace=0.03
)
save_name = (
    f'{time_range[0].strftime("%Y%m%d_%H%M%S")}_{time_range[-1].strftime("%H%M%S")}'
    f'_cinema_ose_{sat_alt=}km_ltan{round(ltan_hours[1])}_{aurora_alt=}km.mp4'
    )
ose.animate_ose(ax=ax, bx=bx, animation_name=save_name, pcolormesh_kwargs={'rasterized':True}, marker=getmarker('camera'), marker_size=200)